# Project 27: EHR with ML, DL and QML

**Team No.:** 3  
**Team Members:** Himanshu Sekhar Behera; Diptesh Mallick; Dibyajyoti Nayak; Guru Gourav Panda  
**Proposed Hybrid Model:** TabTransformer + Variational Quantum Circuit  
**Dataset:** Diabetes 130-US hospitals dataset — https://www.kaggle.com/datasets/brandao/diabetes

Self-contained Google Colab workflow. Run cells top to bottom; outputs are generated from the downloaded data and are intentionally not pre-populated.

## 0. Setup — Environment, Imports, Reproducibility

In [ ]:
!pip -q install kagglehub transformers sentencepiece torch-geometric captum pennylane tqdm tabulate
import os, json, random, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
assert torch.cuda.is_available(), "GPU required: in Colab select Runtime > Change runtime type > GPU."
DEVICE = torch.device("cuda:0")
print('Device:', DEVICE)


### CONFIG

In [ ]:
CONFIG={
 'project_no':'27','project_name':'EHR with ML, DL and QML','team_no':'3','team_members':'Himanshu Sekhar Behera; Diptesh Mallick; Dibyajyoti Nayak; Guru Gourav Panda',
 'task_type':'classification','dataset_name':'Diabetes 130-US hospitals dataset','kaggle_dataset_slug':'brandao/diabetes',
 'proposed_model':'TabTransformer + Variational Quantum Circuit','target_column':None,'id_columns':[],'time_column':None,
 'split_ratios':{'train':.70,'val':.15,'test':.15},'random_seed':SEED,
 'data_raw_dir':'data/raw','data_processed_dir':'data/processed','figures_dir':'figures','results_dir':'results','reports_dir':'reports',
 'batch_size':64,'epochs':20,'patience':4,'learning_rate':1e-3,'max_rows':120000
}
for key in ['data_raw_dir','data_processed_dir','figures_dir','results_dir','reports_dir']: Path(CONFIG[key]).mkdir(parents=True,exist_ok=True)
CONFIG


## 1. Dataset Download

In [ ]:
import kagglehub, shutil
cache_path=Path(kagglehub.dataset_download(CONFIG['kaggle_dataset_slug']))
raw_dir=Path(CONFIG['data_raw_dir'])
for src in cache_path.rglob('*'):
    if src.is_file():
        dst=raw_dir/src.relative_to(cache_path); dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists(): shutil.copy2(src,dst)
raw_files=[p for p in raw_dir.rglob('*') if p.is_file()]
assert raw_files, 'Dataset download produced no files.'
assert sum(p.stat().st_size for p in raw_files)>1024, 'Downloaded payload is unexpectedly small.'
print(f'Discovered {len(raw_files)} files; total bytes={sum(p.stat().st_size for p in raw_files):,}')


## 2. Load Raw Data

In [ ]:
csvs=sorted(Path(CONFIG['data_raw_dir']).rglob('*.csv'),key=lambda p:p.stat().st_size,reverse=True); assert csvs,'No CSV found.'
raw_file=next((p for p in csvs if 'diabetic_data' in p.name.lower()),csvs[0]); df=pd.read_csv(raw_file,na_values=['?','Unknown/Invalid']);
candidates=['readmitted','readmission','target','label']; target=next((c for c in df.columns if c.lower() in candidates),None); assert target is not None,f'Readmission target absent. Columns: {list(df.columns)}'; CONFIG['target_column']=target
if len(df)>CONFIG['max_rows']: df=df.sample(CONFIG['max_rows'],random_state=SEED)
df=df.dropna(subset=[target]).drop_duplicates().reset_index(drop=True); print(raw_file,target,df.shape)


## 3. Exploratory Data Analysis (EDA) + Data Quality Memo

In [ ]:
print('Shape:',df.shape); display(df.head()); print(df.dtypes.value_counts()); print('Duplicates:',int(df.duplicated().sum()))
missing=df.isna().mean().sort_values(ascending=False)
plt.figure(figsize=(9,5)); missing.head(30).sort_values().plot.barh(); plt.xlabel('Missing fraction'); plt.tight_layout(); plt.savefig('figures/fig00_missingness.png',dpi=150); plt.show()
plt.figure(figsize=(7,4)); df[target].astype(str).value_counts().head(30).plot.bar(); plt.title('Target distribution'); plt.tight_layout(); plt.savefig('figures/fig00_target_distribution.png',dpi=150); plt.show()
memo=f"""# Data Quality Memo

- Rows: {len(df):,}; columns: {df.shape[1]}.
- Duplicate rows: {df.duplicated().sum():,}.
- Highest missing fraction: {missing.max():.3f}.
- Target: `{target}` with {df[target].nunique()} observed values.
- Preprocessors are fitted only on training data.
- Dataset-specific leakage controls and schema assertions are applied below.
"""
Path('reports/data_quality_memo.md').write_text(memo,encoding='utf-8'); print(memo)


## 4. Preprocessing & Feature Engineering

The following split cell creates the partitions first and fits all learned preprocessing artifacts on the training partition only.

## 5. Train / Validation / Test Split

In [ ]:
id_col=next((c for c in ['patient_nbr','patient_id'] if c in df.columns),None); CONFIG['id_columns']=[id_col] if id_col else []
if id_col:
 g=GroupShuffleSplit(n_splits=1,train_size=.7,random_state=SEED); ti,ri=next(g.split(df,groups=df[id_col])); train_df,rest=df.iloc[ti].copy(),df.iloc[ri].copy(); g2=GroupShuffleSplit(n_splits=1,train_size=.5,random_state=SEED); vi,si=next(g2.split(rest,groups=rest[id_col])); val_df,test_df=rest.iloc[vi].copy(),rest.iloc[si].copy(); assert set(train_df[id_col]).isdisjoint(test_df[id_col])
else: train_df,rest=train_test_split(df,train_size=.7,stratify=df[target],random_state=SEED); val_df,test_df=train_test_split(rest,train_size=.5,stratify=rest[target],random_state=SEED)
label_encoder=LabelEncoder().fit(train_df[target].astype(str)); known=set(label_encoder.classes_); val_df=val_df[val_df[target].astype(str).isin(known)].copy(); test_df=test_df[test_df[target].astype(str).isin(known)].copy()
drop=[target]+CONFIG['id_columns']; feature_cols=[c for c in df.columns if c not in drop]; num=[c for c in feature_cols if pd.api.types.is_numeric_dtype(train_df[c])]; cat=[c for c in feature_cols if c not in num]
num_imp=SimpleImputer(strategy='median').fit(train_df[num]); scaler=StandardScaler().fit(num_imp.transform(train_df[num])); cat_imp=SimpleImputer(strategy='most_frequent').fit(train_df[cat]); encoder=OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1).fit(cat_imp.transform(train_df[cat]))
def transform(frame):
 continuous=scaler.transform(num_imp.transform(frame[num])).astype('float32')
 categorical=(encoder.transform(cat_imp.transform(frame[cat])).astype('int64')+1)
 return np.hstack([continuous,categorical.astype('float32')])
X_train,X_val,X_test=map(transform,[train_df,val_df,test_df]); y_train=label_encoder.transform(train_df[target].astype(str)); y_val=label_encoder.transform(val_df[target].astype(str)); y_test=label_encoder.transform(test_df[target].astype(str)); feature_names=num+cat; n_classes=len(label_encoder.classes_); n_num=len(num); cat_cardinalities=[len(v)+1 for v in encoder.categories_]
manifest={'train_rows':len(train_df),'val_rows':len(val_df),'test_rows':len(test_df),'group_column':id_col}; Path('data/processed/split_manifest.json').write_text(json.dumps(manifest,indent=2)); print(manifest)


## 6. PyTorch Dataset & DataLoader

In [ ]:
class ArrayDataset(Dataset):
 def __init__(self,x,y): self.x=torch.tensor(x); self.y=torch.tensor(y,dtype=torch.long)
 def __len__(self): return len(self.y)
 def __getitem__(self,i): return self.x[i],self.y[i]
train_loader=DataLoader(ArrayDataset(X_train,y_train),batch_size=CONFIG['batch_size'],shuffle=True); val_loader=DataLoader(ArrayDataset(X_val,y_val),batch_size=CONFIG['batch_size']); test_loader=DataLoader(ArrayDataset(X_test,y_test),batch_size=CONFIG['batch_size'])


## 7. Model Definitions

In [ ]:
import pennylane as qml
assert torch.cuda.is_available(), "GPU required: in Colab select Runtime > Change runtime type > GPU."
DEVICE = torch.device("cuda:0")
class TabTransformerVQC(nn.Module):

    def __init__(self, d, k, n_num, cardinalities, emb=32, qubits=4):
        super().__init__()
        self.n_num = n_num
        self.continuous_tokens = nn.ModuleList([nn.Linear(1, emb) for _ in range(n_num)])
        self.categorical_embeddings = nn.ModuleList([nn.Embedding(card, emb, padding_idx=0) for card in cardinalities])
        layer = nn.TransformerEncoderLayer(emb, 4, 128, batch_first=True, dropout=0.1)
        self.transformer = nn.TransformerEncoder(layer, 2)
        self.to_angles = nn.Linear(emb, qubits)
        dev = qml.device('default.qubit', wires=qubits)

        @qml.qnode(dev, interface='torch', diff_method='backprop')
        def circuit(inputs, weights):
            qml.AngleEmbedding(inputs, wires=range(qubits))
            qml.StronglyEntanglingLayers(weights, wires=range(qubits))
            return [qml.expval(qml.PauliZ(i)) for i in range(qubits)]
        self.quantum = qml.qnn.TorchLayer(circuit, {'weights': (2, qubits, 3)})
        self.head = nn.Linear(emb + qubits, k)

    def to(self, *args, **kwargs):
        super().to(*args, **kwargs)
        self.quantum.to("cpu")  # default.qubit has no CUDA backend
        return self

    def forward(self, x):
        tokens = [layer(x[:, i:i + 1]) for i, layer in enumerate(self.continuous_tokens)]
        tokens.extend((emb(x[:, self.n_num + j].long().clamp(0, emb.num_embeddings - 1)) for j, emb in enumerate(self.categorical_embeddings)))
        h = self.transformer(torch.stack(tokens, 1)).mean(1)
        angles = torch.tanh(self.to_angles(h)) * np.pi
        q = self.quantum(angles.cpu()).to(dtype=h.dtype, device=h.device)
        return self.head(torch.cat([h, q], 1))
hybrid = TabTransformerVQC(X_train.shape[1], n_classes, n_num, cat_cardinalities).to(DEVICE)
with torch.no_grad():
    smoke = hybrid(torch.tensor(X_train[:2], device=DEVICE))
    assert smoke.shape == (2, n_classes) and smoke.device == DEVICE and torch.isfinite(smoke).all()
    assert next(hybrid.quantum.parameters()).device.type == 'cpu', 'default.qubit parameters must remain on CPU.'
loss_fn = lambda model, batch: nn.functional.cross_entropy(model(batch[0]), batch[1])


## 8. Training Loop

In [ ]:
def train_model(model, train_loader, val_loader, loss_fn, checkpoint, epochs=None):
    epochs = epochs or CONFIG['epochs']
    model = model.to(DEVICE)
    grouped = hasattr(model, 'parameter_groups')
    opt = torch.optim.AdamW(model.parameter_groups() if grouped else model.parameters(), lr=CONFIG['learning_rate'], weight_decay=0.0001)
    total_steps = max(1, epochs * len(train_loader))
    warmup_steps = max(1, int(0.1 * total_steps))
    scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lambda step: min((step + 1) / warmup_steps, 1.0)) if grouped else torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', patience=2, factor=0.5)
    best = float('inf')
    stale = 0
    history = {'train_loss': [], 'val_loss': []}
    for epoch in tqdm(range(epochs), desc='Training', unit='epoch'):
        model.train()
        total = 0
        for batch in train_loader:
            batch = [v.to(DEVICE) for v in batch]
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model, batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            if grouped:
                scheduler.step()
            total += loss.item() * batch[0].shape[0]
        train_loss = total / len(train_loader.dataset)
        model.eval()
        total = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = [v.to(DEVICE) for v in batch]
                total += loss_fn(model, batch).item() * batch[0].shape[0]
        val_loss = total / len(val_loader.dataset)
        if not grouped:
            scheduler.step(val_loss)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        print(f'Epoch {epoch + 1:02d}: train={train_loss:.4f}, val={val_loss:.4f}')
        if val_loss < best - 1e-05:
            best = val_loss
            stale = 0
            torch.save(model.state_dict(), checkpoint)
        else:
            stale += 1
            if stale >= CONFIG['patience']:
                break
    model.load_state_dict(torch.load(checkpoint, map_location=DEVICE, weights_only=True))
    return history
hybrid_history = train_model(hybrid, train_loader, val_loader, loss_fn, 'results/best_hybrid.pt')


## 9. Evaluation Metrics

In [ ]:
def predict(model):
    model.eval()
    probs = []
    ys = []
    with torch.no_grad():
        for xb, yb in test_loader:
            probs.append(torch.softmax(model(xb.to(DEVICE)), 1).cpu().numpy())
            ys.append(yb.numpy())
    return (np.vstack(probs), np.concatenate(ys))

def metrics(prob, y):
    pred = prob.argmax(1)
    pr, rc, f1, _ = precision_recall_fscore_support(y, pred, average='macro', zero_division=0)
    out = {'accuracy': float(accuracy_score(y, pred)), 'precision_macro': float(pr), 'recall_macro': float(rc), 'f1_macro': float(f1)}
    try:
        out['roc_auc_ovr_macro'] = float(roc_auc_score(y, prob, multi_class='ovr', average='macro'))
    except ValueError:
        out['roc_auc_ovr_macro'] = None
    return (out, pred)
hybrid_prob, test_y2 = predict(hybrid)
assert np.array_equal(test_y, test_y2)
results = {}
results['hybrid'], hybrid_pred = metrics(hybrid_prob, test_y)
Path('results/metrics.json').write_text(json.dumps(results, indent=2))
print(json.dumps(results, indent=2))


## 10. Required Figures

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(hybrid_history['train_loss'], label='Hybrid train')
plt.plot(hybrid_history['val_loss'], label='Hybrid val')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.tight_layout()
plt.savefig('figures/fig01_loss_curves.png', dpi=150)
plt.show()
cm = confusion_matrix(test_y, hybrid_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('figures/fig02_confusion_matrix.png', dpi=150)
plt.show()
plt.figure(figsize=(8, 5))
for k in tqdm(range(n_classes), desc='Training', unit='epoch'):
    binary = (test_y == k).astype(int)
    if binary.min() != binary.max():
        precision, recall, _ = precision_recall_curve(binary, hybrid_prob[:, k])
        plt.plot(recall, precision, label=str(label_encoder.classes_[k]))
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend(fontsize=7)
plt.tight_layout()
plt.savefig('figures/fig03_precision_recall.png', dpi=150)
plt.show()
sample = next(iter(test_loader))
xb = sample[0][:min(64, len(sample[0]))].to(DEVICE)
hybrid.zero_grad()
xb.requires_grad_(True)
scores = hybrid(xb).max(1).values.sum()
scores.backward()
importance = (xb.grad * xb).abs().mean(0).detach().cpu().numpy()
top = np.argsort(importance)[-20:]
plt.figure(figsize=(8, 5))
plt.barh(np.array(feature_names)[top], importance[top])
plt.tight_layout()
plt.savefig('figures/fig04_feature_importance.png', dpi=150)
plt.show()
errors = (hybrid_pred != test_y).astype(int)
plt.figure(figsize=(8, 4))
pd.Series(errors).rolling(max(5, len(errors) // 40), min_periods=1).mean().plot()
plt.ylabel('Rolling error rate')
plt.tight_layout()
plt.savefig('figures/fig05_error_analysis.png', dpi=150)
plt.show()
metric_names = ['accuracy', 'f1_macro']
x = np.arange(2)
plt.figure(figsize=(6, 4))
plt.xticks(x, metric_names)
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.savefig('figures/fig06_proposed_metrics.png', dpi=150)
plt.show()
